# Data Exploration - Reanalsysis (ERA5 and MERRA2)

In [ ]:
import pathlib
import datetime

In [ ]:
import numpy

In [ ]:
import xarray

In [ ]:
import matplotlib.pyplot
import cartopy

In [ ]:
import site_archive_jasmin

In [ ]:
import pyearthtools.data
import pyearthtools.pipeline


In [ ]:
import torch

### Define common parameters

In [ ]:
select_dt = datetime.datetime(2025,5,11,15,0)

In [ ]:
ghana_extents = {
    'latitude': (4.5,12.3),
    'longitude': (-3.8, 4.0),
}

In [ ]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][0],
                 ghana_extents['longitude'][1],
                )

### MERRA2 dataset info
Reanalysis dataset used for aerosols
* dir location `/gws/nopw/j04/ew4energy/West_Africa_Merra-2pwd`
* file template `M2T1NXAER.5.12.4%3AMERRA2_401.tavg1_2d_aer_Nx.20200930.nc4.dap.nc4` 
* data time span 2015-2020
* data temporal resolution: 1 day per file


In [ ]:
ew4_dir = pathlib.Path('/gws/nopw/j04/ew4energy/')

In [ ]:
meteo_fname_template = 'MERRA2_400.tavg1_2d_slv_Nx.{dt.year:04d}{dt.month:02d}{dt.day:02d}.SUB.nc'
aero_fname_template = 'MERRA2_400.inst3_3d_aer_Nv.{dt.year:04d}{dt.month:02d}{dt.day:02d}.SUB.nc'

In [ ]:
merra2_meteo_dir = ew4_dir / 'West_Africa_Merra-2_2015-2025_meteo' 
merra2_meteo_dir

In [ ]:
merra2_aero_dir = ew4_dir / 'West_Africa_Merra-2_2015-2025_aerosols' / '3d'
merra2_aero_dir

### Explore the meteological data variables.

In [ ]:
merra2_meteo_sample = xarray.open_dataset(merra2_meteo_dir / f'{select_dt.year:04d}' / meteo_fname_template.format(dt=select_dt))

In [ ]:
float(merra2_meteo_sample['lat'].min()), float(merra2_meteo_sample['lat'].max()), float(merra2_meteo_sample['lon'].min()), float(merra2_meteo_sample['lon'].max())

In [ ]:
merra2_meteo_sample

In [ ]:
list(merra2_meteo_sample.variables)

In [ ]:
float(min(merra2_meteo_sample['lat'])), float(max(merra2_meteo_sample['lat'])), float(min(merra2_meteo_sample['lon'])), float(max(merra2_meteo_sample['lon'])),

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
merra2_meteo_sample['T2M'][10].plot.contourf(ax=ax1)
ax1.coastlines(color='w')
merra2_meteo_sample['T2M']

### Explore the aerosol data variables

In [ ]:
merra2_aero_sample = xarray.open_dataset(merra2_aero_dir / f'{select_dt.year:04d}' / aero_fname_template.format(dt=select_dt))

In [ ]:
merra2_aero_sample

In [ ]:
list(merra2_aero_sample.variables)

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
merra2_aero_sample['SO2'][0][0].plot.contourf(ax=ax1)
ax1.coastlines(color='w')


If we merge the two datasets, we can see what this will look like. This might the sort of dataset that we want to get out of our loader.

In [ ]:
xarray.merge([merra2_aero_sample['SO2'], merra2_meteo_sample['U10M'], merra2_aero_sample['SO4'], merra2_meteo_sample['V10M']])

## Load the data in pyearthtools

In [ ]:
ew4_merra2_aero_accessor = pyearthtools.data.archive.ew4_merra2_aero(['SO2', 'SO4'])
ew4_merra2_aero_accessor

In [ ]:
ew4_merra2_aero_accessor[select_dt]

In [ ]:
ew4_merra2_meteo_accessor = pyearthtools.data.archive.ew4_merra2_meteo(['T2M', 'U2M'])
ew4_merra2_meteo_accessor

In [ ]:
ew4_merra2_meteo_accessor[select_dt]

In [ ]:
ew4_merra_load_pipe = pyearthtools.pipeline.Pipeline(
    (ew4_merra2_aero_accessor, ew4_merra2_meteo_accessor),
    pyearthtools.pipeline.operations.xarray.Merge(),
)
ew4_merra_load_pipe[select_dt]



In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
ew4_merra_load_pipe['2025-04-01 15:00']['U2M'][0].plot.contourf(ax=ax1)
ax1.coastlines(color='w')

In [ ]:
ew4_merra2_pipeline = pyearthtools.pipeline.Pipeline(
    (ew4_merra2_aero_accessor, ew4_merra2_meteo_accessor),
    pyearthtools.pipeline.operations.xarray.Merge(),
    # pyearthtools.data.transform.region.Bounding(*ghana_pet_box),  
    pyearthtools.pipeline.operations.xarray.select.SliceDataset(slices=ghana_extents),
    pyearthtools.pipeline.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=pyearthtools.data.time.TimeDelta('30 minutes')),
    iterator=pyearthtools.pipeline.iterators.DateRange('20250401T00', '20250901T00', interval='3 hours'), 
    exceptions_to_ignore=pyearthtools.data.exceptions.DataNotFoundError,
)

In [ ]:
predictor, target = ew4_merra2_pipeline[select_dt]

In [ ]:
predictor[0]

In [ ]:
target[0]

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
target[0]['T2M'][0].plot.contourf(ax=ax1)
ax1.coastlines(color='w')

In [ ]:
pred_n, target_n = next(iter(ew4_merra2_pipeline))

In [ ]:
pred_n[0]

In [ ]:
target_n[0]

# ERA5 Reanalsyis Dataset

### ERA5 Intial Exploration
For the purposes of this tutorial, I have downloaded a subset from the Copernicus Climate Data Store to demonstrate its use.

In [ ]:
era5_ew4_dir = ew4_dir / 'ERA5' / 'tutorial_202606'

In [ ]:
era5_fname_template = 'era5_pl_{dt.year:04d}_{dt.month:02d}_vertical_velocity.nc'

In [ ]:
select_dt = datetime.datetime(2025,7,23,9,0)

In [ ]:
era5_sample = xarray.open_dataset(era5_ew4_dir / era5_fname_template.format(dt=select_dt))

In [ ]:
era5_sample

In [ ]:
float(era5_sample['latitude'].min()), float(era5_sample['latitude'].max()), float(era5_sample['longitude'].min()), float(era5_sample['longitude'].max())

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
era5_sample['w'][0][0].plot.contourf(ax=ax1)
ax1.coastlines(color='w')


### Load in PyEarthTools

As with the other datasets, we can create a dataaet accessor to abstract away the details of accessing the data.

In [ ]:
ew4_era5_accessor = pyearthtools.data.archive.ew4_era5(variables=['temperature','specific_humidity', 'vertical_velocity'])

In [ ]:
ew4_era5_accessor[select_dt]

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
ew4_era5_accessor[select_dt].sel(pressure_level=500)['w'].squeeze().plot.contourf(ax=ax1)
ax1.coastlines(color='w')

